In [ ]:
import cv2
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')
ruta_base = '/content/drive/MyDrive/Proyecto_arroz'

def generar_dataset():
    matriz_final = []
    config = {'Positivo': 1, 'Negativo': 0}

    for nombre_carpeta, etiqueta in config.items():
        # Usamos ruta_especifica para buscar los archivos
        ruta_especifica = os.path.join(ruta_base, nombre_carpeta)

        if not os.path.exists(ruta_especifica):
            print(f"La carpeta {nombre_carpeta} no existe en {ruta_base}")
            continue

        archivos = [f for f in os.listdir(ruta_especifica) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Procesando {len(archivos)} imágenes de la clase '{nombre_carpeta}'...")

        for nombre in archivos:
            path_completo = os.path.join(ruta_especifica, nombre)
            img = cv2.imread(path_completo, cv2.IMREAD_GRAYSCALE)

            if img is not None:
                img_res = cv2.resize(img, (128, 128))

                # Binarización Otsu + Inversión (Fondo=1, Objeto=0)
                _, seg = cv2.threshold(img_res, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                img_inv = cv2.bitwise_not(seg)
                img_bin = (img_inv / 255).astype(np.uint8)

                # Vectorizar y añadir etiqueta
                fila = np.append(img_bin.flatten(), etiqueta)
                matriz_final.append(fila)

    if matriz_final:
        dataset = np.array(matriz_final)

        # 3. Guardar CSV con encabezados
        columnas = [f'pixel_{i}' for i in range(128*128)] + ['etiqueta_arroz']
        encabezado = ",".join(columnas)

        ruta_csv = os.path.join(ruta_base, 'matriz_final.csv')
        np.savetxt(ruta_csv, dataset, delimiter=",", fmt='%d', header=encabezado, comments='')

        print(f"\nPROCESO EXITOSO")
        print(f"Dimensiones: {dataset.shape}")
        print(f"Archivo guardado en Drive: {ruta_csv}")
    else:
        print("No se encontraron imágenes. Revisa las carpetas 'Positivo' y 'Negativo'.")

# Ejecutamos la función
generar_dataset()